In [ ]:
import scipy.stats as st
import scanpy as sc

### Manual annotation by just inspecting the expression of marker genes

In [ ]:
sc.pl.umap(
    adata,
    color="CD4",
    cmap="viridis",
    title="CD4 expression"
)

### Manual annotation by just inspecting the expression of top degs per cluster (possible marker genes)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden_r_0.5", method="wilcoxon")
degs=sc.get.rank_genes_groups_df(adata,group=None, pval_cutoff = 0.01)
degs

In [ ]:
top2 = (degs.groupby("group").head(2))
top2

In [ ]:
genes_to_plot = top2["names"].unique().tolist()
genes_to_plot

In [ ]:
sc.pl.umap(
    adata,
    color=genes_to_plot,
    cmap="viridis",
    ncols=2
)

In [ ]:
sc.pl.stacked_violin(adata,genes_to_plot, groupby="leiden_r_0.5", swap_axes=True)

### Based on Enrichment how could we do the annotation? 

In [ ]:
def readAnnotations():
    data = {}
    class_names = {}
    universe = set()

    with open("sctype_annotations.csv") as f:
        for line in f:
            line = line.strip().split("|")
            gene = line[0]
            class_id = line[1]
            name = line[2]

            universe.add(gene)

            if class_id not in data:
                data[class_id] = set()

            if class_id not in class_names:
                class_names[class_id] = name

            data[class_id].add(gene)

    return data, class_names, universe

In [ ]:
def readGenes():
    genelist=set()
    f=open('/degs_genes.txt','r')
    for line in f:
        gene=line.strip()
        genelist.add(gene)
        universe.add(gene)
    f.close()

    return genelist

In [ ]:
degs["names"].dropna().unique().tofile(
    "degs_genes.txt",
    sep="\n"
)

In [ ]:
classes, class_names, universe = readAnnotations()
genelist = readGenes()
universe=universe | genelist

for cat in classes:
    Cl=classes[cat]
    a=len(genelist & Cl)
    b=len(genelist - Cl)
    c=len(Cl-genelist)
    d=len(universe-genelist-Cl)
    
    matrix=[[a,b],[c,d]]
    pvalue=st.fisher_exact(matrix,'greater')
    print(cat,class_names[cat],pvalue[1])